# Polymarket Data Toolkit
### Query Postgres → save locally → join with order books and resolutions

This notebook is a set of **reusable functions** for building your own analysis dataset. Each function is explained just above its code.

**The pipeline**

```
 1. get_markets()            Postgres markets/events  ──► data/<name>.parquet
 2. get_resolutions()        Postgres resolutions     ──► data/<name>_resolutions.parquet
 3. join_resolutions()       markets + who won
 4. token_table()            one row per YES/NO token (the order-book join key)
 5. orderbook_snapshots()    hourly best bid/ask per token  ──► data/<name>_snapshots.parquet
    orderbook_deltas()       every book change per token    ──► data/<name>_deltas.parquet
 6. build_panel()            one row per market per hour: price + outcome
```

**Every function saves its result** to the `data/` folder. Run it again and it reads the saved file instead of querying. So:
- re-running the notebook is fast, and
- **it keeps working after the access window closes (Oct 29 2026, 11:59pm PDT).**

Pass `refresh=True` to any function to query again and overwrite the saved file.

**How the datasets join**

| From | Column | To | Column |
|---|---|---|---|
| `markets` | `slug` | `resolutions` | `market_slug` |
| `markets` | `condition_id` | order-book files | `market` |
| `markets` | each id in `token_ids` | order-book files | `asset_id` |
| `markets` | `slug` | wallet-trade files | `market` |

---
## 1 · Setup

### 1.1 Install packages (run once per environment)

In [ ]:
%pip install --quiet pandas requests pyarrow sqlalchemy psycopg2-binary duckdb matplotlib

### 1.2 Settings and connections
- **Postgres** (market metadata) is reached through SQLAlchemy. `quote()` makes the `!` in the password safe inside the connection string, and `pool_pre_ping=True` reconnects quietly if Supabase dropped an idle connection.
- **DuckDB** reads the Parquet files on CloudFront. The login travels as an `Authorization` header stored in a DuckDB SECRET, scoped to the CloudFront address only.
- `SET TimeZone = 'UTC'` keeps every timestamp in UTC, matching the data.

In [1]:
import base64
import json
from pathlib import Path
from urllib.parse import quote

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text

# ---- Course logins ---------------------------------------------------------
USER = "polymarketdatabase"
PASSWORD = "PolymarketData2026!"
PG_USER = f"{USER}.merbqatlxbrcvtbnuaqe"   # the suffix is PART of the username
PG_HOST, PG_PORT = "aws-0-us-west-2.pooler.supabase.com", 6543
BASE = "https://d1yyqbluk0oz0t.cloudfront.net"   # all Parquet files live here

DATA_DIR = Path("data")      # every saved result goes in this folder
DATA_DIR.mkdir(exist_ok=True)

# ---- Postgres --------------------------------------------------------------
engine = create_engine(
    f"postgresql+psycopg2://{quote(PG_USER, safe='')}:{quote(PASSWORD, safe='')}"
    f"@{PG_HOST}:{PG_PORT}/postgres?sslmode=require",
    pool_pre_ping=True,
)

# ---- DuckDB ----------------------------------------------------------------
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("SET TimeZone = 'UTC'")
token = base64.b64encode(f"{USER}:{PASSWORD}".encode()).decode()
con.execute(f'''
    CREATE OR REPLACE SECRET polymarket_cdn (
        TYPE http,
        SCOPE '{BASE}/',
        EXTRA_HTTP_HEADERS MAP {{'Authorization': 'Basic {token}'}}
    )
''')
print("ready")

ready


---
## 2 · Querying Postgres and saving the result

### 2.1 `sql(query, **params)`: run a query
Returns a pandas DataFrame. Pass values as keyword arguments and refer to them as `:name` in the SQL. This is safer than pasting values into the string, and it handles lists: `WHERE slug = ANY(:slugs)` with `slugs=[...]`.

### 2.2 `save(df, name)` / `load(name)`: store a DataFrame in `data/`
Parquet keeps column types (dates stay dates), unlike CSV. The JSON list columns `token_ids` and `outcomes` are stored as JSON text, because Parquet can't store mixed Python lists reliably; `as_list()` turns them back into lists.

### 2.3 `sql_saved(name, query, refresh=False, **params)`: query once, then reuse
The first call runs the query and saves it to `data/<name>.parquet`. Every later call reads that file instead, even after the deadline.

In [ ]:
def sql(query, **params):
    '''Run SQL on the Postgres database and return a DataFrame.'''
    with engine.connect() as c:
        return pd.read_sql(text(query), c, params=params)


JSON_COLS = ("token_ids", "outcomes")

def as_list(x):
    '''A JSON column value -> Python list (works for lists and JSON text).'''
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []
    return json.loads(x) if isinstance(x, str) else list(x)

def save(df, name):
    '''Save a DataFrame to data/<name>.parquet and return it.'''
    out = df.copy()
    for col in JSON_COLS:
        if col in out:
            out[col] = out[col].map(lambda v: json.dumps(as_list(v)))
    path = DATA_DIR / f"{name}.parquet"
    out.to_parquet(path, index=False)
    print(f"saved {path}: {len(out):,} rows")
    return out

def load(name):
    '''Read data/<name>.parquet back into a DataFrame.'''
    return pd.read_parquet(DATA_DIR / f"{name}.parquet")

def is_saved(name):
    return (DATA_DIR / f"{name}.parquet").exists()

def sql_saved(name, query, refresh=False, **params):
    '''Run a Postgres query once and save it; later calls read the saved file.'''
    if is_saved(name) and not refresh:
        return load(name)
    return save(sql(query, **params), name)

### 2.4 `get_markets(name, where, **params)`: pick your markets
You write only the `WHERE` part. The function always includes the columns needed for later joins (`slug`, `condition_id`, `token_ids`, `outcomes`), plus the event's title. Use the alias `m.` for market columns and `e.` for event columns.

Examples of `where`:
- `"e.title ILIKE :pat"` with `pat="%midterm%"`: markets in events whose title mentions "midterm"
- `"m.volume_1wk > 100000 AND m.active"`: busy, currently open markets
- `"m.end_date BETWEEN '2026-09-01' AND '2026-10-01'"`: markets scheduled to close in September

In [ ]:
MARKET_COLS = '''
    m.slug, m.title, m.condition_id, m.event_slug, e.title AS event_title,
    m.token_ids, m.outcomes, m.neg_risk,
    m.start_date, m.end_date, m.active, m.closed,
    m.best_bid, m.best_ask, m.last_trade_price,
    m.volume_clob, m.volume_1wk
'''

def get_markets(name, where="TRUE", refresh=False, limit=None, **params):
    '''Markets matching a WHERE clause, saved as data/<name>.parquet.'''
    query = f'''
        SELECT {MARKET_COLS}
        FROM markets m
        LEFT JOIN events e ON e.slug = m.event_slug
        WHERE {where}
        ORDER BY m.volume_clob DESC NULLS LAST
        {f"LIMIT {int(limit)}" if limit else ""}
    '''
    return sql_saved(name, query, refresh=refresh, **params)

---
## 3 · Joining with resolutions

### 3.1 `get_resolutions(markets, name)`: who won, for your markets only
Fetches the `resolutions` rows for the slugs in your markets table and saves them as `data/<name>_resolutions.parquet`.

### 3.2 `join_resolutions(markets, res)`: add the outcome to each market
A **left join**: every market stays, even unresolved ones (their outcome columns are empty). The function adds:
- `resolved`: `True` if a resolution row exists;
- `outcome_yes`: 1 if YES won, 0 if NO won, empty if not resolved yet.

> A market with a past `end_date` and no resolution is **still open** (e.g. waiting for the UMA oracle). It isn't missing data.

In [ ]:
def get_resolutions(markets, name, refresh=False):
    '''resolutions rows for the markets in `markets`, saved as data/<name>_resolutions.parquet.'''
    return sql_saved(f"{name}_resolutions", '''
        SELECT market_slug, resolved_yes, price_at_close, resolved_at, uma_resolution_status
        FROM resolutions
        WHERE market_slug = ANY(:slugs)
    ''', refresh=refresh, slugs=markets["slug"].tolist())

def join_resolutions(markets, res):
    '''Markets + their resolution (left join on slug).'''
    out = markets.merge(res, left_on="slug", right_on="market_slug", how="left")
    out = out.drop(columns="market_slug")
    out["resolved"] = out["resolved_yes"].notna()
    out["outcome_yes"] = out["resolved_yes"].map({True: 1, False: 0})
    return out

---
## 4 · Joining with order books

### 4.1 `token_table(markets)`: the join key
The order-book files list each side of a market by its **token id** (`asset_id`), not by slug. A market's `token_ids` holds two ids, YES first then NO, and `outcomes` holds the matching labels.
`token_table` turns each market into **two rows**, one per token, so it can be joined to the order books on `asset_id`.

| slug | condition_id | asset_id | outcome | is_yes |
|---|---|---|---|---|
| will-x-happen | 0xabc… | 1234… | Yes | True |
| will-x-happen | 0xabc… | 5678… | No | False |

### 4.2 `hour_keys(start, end, kind)`: file names for a range of hours
Order-book files are hourly (`snap-2026-05-30T21.parquet`, `events-2026-05-30T21.parquet`), with hours in **UTC**. This builds the list of names and **skips the three known recorder gaps**. Listing names is necessary because wildcards like `*` don't work over HTTPS.

In [ ]:
def token_table(markets):
    '''One row per token: slug, condition_id, asset_id, outcome, is_yes.'''
    rows = []
    for m in markets.itertuples():
        tokens, labels = as_list(m.token_ids), as_list(m.outcomes)
        for i, (asset, label) in enumerate(zip(tokens, labels)):
            rows.append({"slug": m.slug, "condition_id": m.condition_id,
                         "asset_id": str(asset), "outcome": label, "is_yes": i == 0})
    return pd.DataFrame(rows, columns=["slug", "condition_id", "asset_id", "outcome", "is_yes"])


# Hours (UTC) when the recorder was down: no files, permanently.
GAPS = [("2026-06-16T11", "2026-06-19T19"),
        ("2026-08-25T00", "2026-08-31T14"),
        ("2026-09-14T01", "2026-09-14T15")]
FIRST_HOUR = "2026-05-30T00"   # capture started 2026-05-30

def in_gap(hour):
    return any(a <= hour <= b for a, b in GAPS)

def hour_keys(start, end, kind):
    '''File names for every hour start..end (inclusive). kind = "snap" or "events".'''
    hours = pd.date_range(start, end, freq="h").strftime("%Y-%m-%dT%H")
    return [f"{kind}-{h}.parquet" for h in hours if h >= FIRST_HOUR and not in_gap(h)]

def path_for(key):
    '''Local copy in data/ if you saved the raw file, otherwise its URL.'''
    local = DATA_DIR / key
    return local.as_posix() if local.exists() else f"{BASE}/{key}"

### 4.3 `query_files(keys, sql_template)`: run one query per file, skip missing ones
Reading a list of files in a single query fails completely if just **one** hour is missing. This helper queries the files **one at a time**: it prints and skips any file that fails, then stacks the results.

Inside DuckDB, the token table is available as a table called `tok` (`con.register` lets SQL use a pandas DataFrame). Joining to it **filters each big hourly file down to your markets' tokens** before anything reaches pandas.

> ⚠️ If *every* file fails, check the error message: 401/403 means a login problem, or that you're outside the access window.

In [ ]:
def query_files(keys, sql_template, label="files"):
    '''Run sql_template (with {src} = file path) on each file; skip failures; concat.'''
    parts, failed = [], []
    for i, key in enumerate(keys, 1):
        try:
            parts.append(con.sql(sql_template.format(src=path_for(key))).df())
        except duckdb.Error as e:
            failed.append(key)
            print(f"  skipped {key}: {str(e).splitlines()[0][:120]}")
        if i % 24 == 0:
            print(f"  {label}: {i}/{len(keys)} done")
    if failed and len(failed) == len(keys):
        raise RuntimeError(f"All {len(keys)} {label} failed; see the messages above.")
    parts = [p for p in parts if len(p)]
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

### 4.4 `orderbook_snapshots(markets, name, start, end)`: hourly top of book
Each snapshot file has the **full book at the start of the hour** for every token. For your markets' tokens, this function keeps:

| column | meaning |
|---|---|
| `hour` | start of the hour (UTC) |
| `best_bid` / `best_ask` | highest bid / lowest ask (`bids[1]` / `asks[1]`; lists are best-first, and DuckDB counts from 1) |
| `bid_size` / `ask_size` | shares waiting at the best bid / ask |
| `bid_levels` / `ask_levels` | number of price levels on each side (a rough depth measure) |
| `mid`, `spread` | (bid + ask) / 2 and ask − bid |

…joined with `token_table`, so each row also has `slug`, `outcome` and `is_yes`. Saved as `data/<name>_snapshots.parquet`.

> ⚠️ The snapshot in the **first hour after a recorder gap** may be stale (from before the gap). Treat those hours with care, or drop them.

In [ ]:
SNAP_SQL = '''
    SELECT epoch_ms(s.ts_ms)          AS hour,
           t.slug, t.outcome, t.is_yes, s.asset_id,
           s.bids[1].price            AS best_bid,
           s.bids[1].size             AS bid_size,
           s.asks[1].price            AS best_ask,
           s.asks[1].size             AS ask_size,
           len(s.bids)                AS bid_levels,
           len(s.asks)                AS ask_levels
    FROM '{src}' s
    JOIN tok t ON t.asset_id = s.asset_id
'''

def orderbook_snapshots(markets, name, start, end, refresh=False):
    '''Hourly best bid/ask for every token of `markets`, start..end (UTC hours).'''
    key = f"{name}_snapshots"
    if is_saved(key) and not refresh:
        return load(key)
    con.register("tok", token_table(markets))
    keys = hour_keys(start, end, "snap")
    print(f"reading {len(keys)} snapshot files…")
    snaps = query_files(keys, SNAP_SQL, label="snapshots")
    if snaps.empty:
        print("No rows matched. Check that your markets were trading in this period "
              "(see check_ids() below).")
        return snaps
    snaps["mid"] = (snaps["best_bid"] + snaps["best_ask"]) / 2
    snaps["spread"] = snaps["best_ask"] - snaps["best_bid"]
    return save(snaps.sort_values(["slug", "outcome", "hour"]), key)

### 4.5 `orderbook_deltas(markets, name, start, end)`: every book change
The delta files have **one row per price-level change**: `side` (`BUY` = bid, `SELL` = ask), `price`, and the **new total** `size` at that price (`0` = level removed).
Use this when hourly snapshots are too coarse, e.g. to rebuild the book at an exact moment or to study reactions to news. It can be **many** rows for busy markets, so start with a short time range.

In [ ]:
DELTA_SQL = '''
    SELECT epoch_ms(d.ts_ms) AS time, d.ts_ms,
           t.slug, t.outcome, t.is_yes, d.asset_id,
           d.side, d.price, d.size
    FROM '{src}' d
    JOIN tok t ON t.asset_id = d.asset_id
'''

def orderbook_deltas(markets, name, start, end, refresh=False):
    '''Every order-book change for the tokens of `markets`, start..end (UTC hours).'''
    key = f"{name}_deltas"
    if is_saved(key) and not refresh:
        return load(key)
    con.register("tok", token_table(markets))
    keys = hour_keys(start, end, "events")
    print(f"reading {len(keys)} delta files…")
    deltas = query_files(keys, DELTA_SQL, label="delta files")
    if deltas.empty:
        print("No rows matched.")
        return deltas
    return save(deltas.sort_values(["slug", "outcome", "ts_ms"]), key)

### 4.6 `check_ids(markets, hour)`: troubleshooting an empty result
If the order-book functions return nothing, either your markets weren't trading in those hours, or the ids don't line up. This prints a few `asset_id`s from one file next to a few from your `token_ids`, so you can compare their format, and counts how many of your tokens appear in that file.

In [ ]:
def check_ids(markets, hour):
    '''Compare token ids in your markets with asset_ids in one snapshot file.'''
    tok = token_table(markets)
    file_ids = con.sql(f"SELECT DISTINCT asset_id FROM '{path_for(f'snap-{hour}.parquet')}'").df()
    print("from the file :", file_ids["asset_id"].head(3).tolist())
    print("from markets  :", tok["asset_id"].head(3).tolist())
    found = tok["asset_id"].isin(file_ids["asset_id"]).sum()
    print(f"{found} of your {len(tok)} tokens appear in snap-{hour}.parquet")

---
## 5 · `build_panel()`: putting it together

`build_panel` makes **one row per market per hour**, ready for analysis:
1. keeps the **YES** token's snapshot rows (the NO token's price is about 1 − YES, so it adds little);
2. joins the market's details and **resolution** on `slug`;
3. adds `hours_to_resolution`: how long before the market resolved this price was observed.

With that you can ask things like: *how accurate was the price 24 hours before resolution?*

In [ ]:
def build_panel(markets_res, snaps):
    '''One row per market-hour: YES-token top of book + market info + outcome.'''
    yes = snaps[snaps["is_yes"]].drop(columns=["is_yes", "outcome"])
    keep = ["slug", "title", "event_title", "end_date", "resolved", "outcome_yes", "resolved_at"]
    panel = yes.merge(markets_res[keep], on="slug", how="left")
    panel["hours_to_resolution"] = (
        (pd.to_datetime(panel["resolved_at"], utc=True)
         - pd.to_datetime(panel["hour"], utc=True)).dt.total_seconds() / 3600
    )
    return panel.sort_values(["slug", "hour"]).reset_index(drop=True)

---
## 6 · Worked example

Each step saves its result, so after the first run the whole section loads from `data/` in seconds.

### Step 1: choose markets
Here: markets that closed between Sept 15 and Oct 5 2026, inside the order-book coverage, with real trading volume. Change the `where` to fit your question.

In [ ]:
NAME = "sept_markets"     # prefix for every saved file in this example

mkts = get_markets(
    NAME,
    where="m.end_date BETWEEN :start AND :end AND m.volume_clob > :min_vol",
    start="2026-09-15", end="2026-10-05", min_vol=50_000,
    limit=200,
)
print(f"{len(mkts)} markets")
mkts[["slug", "title", "end_date", "volume_clob"]].head()

### Step 2: add resolutions

In [ ]:
res = get_resolutions(mkts, NAME)
mkts_res = join_resolutions(mkts, res)

print(mkts_res["resolved"].value_counts().rename({True: "resolved", False: "not yet"}))
mkts_res[["title", "resolved", "outcome_yes", "price_at_close", "resolved_at"]].head()

### Step 3: order-book snapshots
One file per hour, so a 3-day range reads 72 files. Start small, then widen it.
If you get nothing back, run `check_ids(mkts, "2026-10-01T12")`.

In [ ]:
snaps = orderbook_snapshots(mkts, NAME, start="2026-10-01T00", end="2026-10-03T23")
snaps.head()

### Step 4: the analysis panel

In [ ]:
panel = build_panel(mkts_res, snaps)
save(panel, f"{NAME}_panel")
panel[["hour", "title", "mid", "spread", "outcome_yes", "hours_to_resolution"]].head(10)

### Step 5: a quick look at the price before resolution vs. the outcome
For each resolved market, take the **last YES midpoint at least 24 hours before resolution**, then compare markets where YES won with markets where NO won. A market with good prices should show **high midpoints for YES winners and low ones for NO winners**.

In [ ]:
before = (panel[panel["resolved"] & (panel["hours_to_resolution"] >= 24) & panel["mid"].notna()]
          .sort_values("hour")
          .groupby("slug").tail(1))           # last observation per market

print(f"{len(before)} resolved markets with a price ≥ 24h before resolution")
print(before.groupby("outcome_yes")["mid"].describe()[["count", "mean", "50%"]])

fig, ax = plt.subplots(figsize=(8, 4))
bins = np.linspace(0, 1, 21)
ax.hist(before.loc[before["outcome_yes"] == 1, "mid"], bins=bins, alpha=0.75,
        color="#2a78d6", edgecolor="white", label="YES won")
ax.hist(before.loc[before["outcome_yes"] == 0, "mid"], bins=bins, alpha=0.75,
        color="#eb6834", edgecolor="white", label="NO won")
ax.set_xlabel("YES midpoint ≥ 24h before resolution")
ax.set_ylabel("number of markets")
ax.legend(frameon=False)
ax.grid(alpha=0.25, lw=0.5, axis="y")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

brier = ((before["mid"] - before["outcome_yes"]) ** 2).mean()
print(f"Brier score: {brier:.3f}  (0 = perfect; always saying 50% scores 0.250)")

### What's in `data/` now

In [ ]:
for p in sorted(DATA_DIR.glob("*.parquet")):
    print(f"{p.name:45s} {p.stat().st_size / 1e6:8.2f} MB")

### Helper functions for analyzing the wallet data

In [ ]:
def trade_keys(start_date, end_date):
    '''Wallet-trade file names, one per day (UTC), start_date..end_date inclusive.'''
    days = pd.date_range(start_date, end_date, freq="D").strftime("%Y-%m-%d")
    return [f"wallet_trades/trades-{d}.parquet" for d in days]

def wallet_trades(markets, name, start_date, end_date, refresh=False):
    '''All trades in `markets` between two dates, saved as data/<name>_trades.parquet.'''
    key = f"{name}_trades"
    if is_saved(key) and not refresh:
        return load(key)
    con.register("mk", markets[["slug"]])
    trades = query_files(trade_keys(start_date, end_date),
                         "SELECT t.* FROM '{src}' t JOIN mk ON mk.slug = t.market",
                         label="trade files")
    return save(trades, key)

def wallet_trades_by_wallets(wallets, name, start_date, end_date, refresh=False):
    '''All trades (in ANY market) by the given wallets, saved as data/<name>.parquet.'''
    if is_saved(name) and not refresh:
        return load(name)
    con.register("ww", pd.DataFrame({"wallet": pd.unique(wallets)}))
    trades = query_files(trade_keys(start_date, end_date),
                         "SELECT t.* FROM '{src}' t JOIN ww ON ww.wallet = t.wallet",
                         label="trade files")
    return save(trades, name)

def gini(values):
    '''0 = everyone trades the same amount, 1 = one wallet does everything.'''
    v = np.sort(np.asarray(values, dtype=float))
    n = len(v)
    return (2 * np.sum(np.arange(1, n + 1) * v) / (n * v.sum())) - (n + 1) / n

def trader_metrics(trades):
    '''The trader-behavior table for one group of trades.'''
    per_wallet = trades.groupby("wallet").agg(
        volume=("dollars", "sum"), n_trades=("dollars", "size"), n_markets=("market", "nunique"))
    top_n = max(1, int(np.ceil(len(per_wallet) * 0.01)))     # how many wallets make the top 1%
    return pd.Series({
        "Unique traders (wallets)":               len(per_wallet),
        "Share of volume from top 1% of wallets": per_wallet["volume"].nlargest(top_n).sum() / per_wallet["volume"].sum(),
        "Gini coefficient (volume per wallet)":   gini(per_wallet["volume"]),
        "% of trades under $10":                  (trades["dollars"] < 10).mean(),
        "% of trades over $1,000":                (trades["dollars"] > 1000).mean(),
        "% one-time traders (1 trade)":           (per_wallet["n_trades"] == 1).mean(),
        "Markets per trader, median":             per_wallet["n_markets"].median(),
    })

---
## 7 · Using the saved files after the deadline

After **Oct 29 2026, 11:59pm PDT** the Postgres and CloudFront logins stop working, but everything you saved still loads:

```python
mkts     = load("sept_markets")
res      = load("sept_markets_resolutions")
snaps    = load("sept_markets_snapshots")
panel    = load("sept_markets_panel")
```

Calling the functions with the same `name` also works, because they read the saved file first. Only `refresh=True`, a **new** name, or a wider time range will try to reach the servers.

You can also run SQL on saved files with DuckDB:
```python
con.sql("SELECT slug, AVG(spread) FROM 'data/sept_markets_snapshots.parquet' GROUP BY slug").df()
```

**Saved metadata is a snapshot.** `active`, `best_bid` and the volume columns are frozen on the day you saved them; note that date in your write-up.